In [2]:
import crypten
import torch
import crypten.communicator as comm
import crypten.mpc as mpc


crypten.init() 
torch.set_num_threads(1)

# Ignore warnings
import warnings; 
warnings.filterwarnings("ignore")

# Keep track of all created temporary files so that we can clean up at the end
temp_files = []

In [3]:
ALICE = 0
BOB = 1

a = torch.Tensor([2, 4, 5 ,8])
a += torch.randn(a.shape)
torch.save(a,"/tmp/my_experiments/p_0.pt")



# Bob
b = torch.Tensor([3,7,9,10])
b =+ torch.randn(b.shape)
torch.save(b,"/tmp/my_experiments/p_1.pt")

a,b




(tensor([1.2278, 5.3980, 4.7446, 8.2197]),
 tensor([ 0.0620, -0.2695, -1.1766, -0.7306]))

In [36]:
@mpc.run_multiprocess(world_size=2)
def run_collaborative_mean():
    #already noised coeffiecients
    
    a_enc = crypten.load_from_party("/tmp/my_experiments/p_0.pt", src=ALICE)
    b_enc = crypten.load_from_party("/tmp/my_experiments/p_1.pt", src=BOB)
   
    rank = comm.get().get_rank()
    
    boh = torch.load(f"/tmp/my_experiments/p_{rank}.pt")
    crypten.print(f"Rank: {rank}\n\t boh : {boh}", in_order=True)
    
    crypten.print(f"Rank: {rank}\n\ta share: {a_enc}", in_order=True)
    crypten.print(f"Rank: {rank}\n\tb share:{b_enc}", in_order=True)


    agg = (a_enc + b_enc)*0.5


    crypten.print(f"Rank: {rank}\n\taggregated:{agg}", in_order=True)
    crypten.print(f"Rank: {rank}\n\taggregated:{agg.get_plain_text()}", in_order=True)

run_collaborative_mean()

Rank: 0
	 boh : tensor([2.3136, 3.3740, 3.0456, 9.4094])
Rank: 1
	 boh : tensor([0.3703, 3.0630, 0.6136, 1.1902])
Rank: 0
	a share: MPCTensor(
	_tensor=tensor([-9055064864172800399,  6723021810210044015,   337440753389374521,
         7741897742298624379])
	plain_text=HIDDEN
	ptype=ptype.arithmetic
)
Rank: 1
	a share: MPCTensor(
	_tensor=tensor([ 9055064864172952020, -6723021810209822900,  -337440753389174926,
        -7741897742298007724])
	plain_text=HIDDEN
	ptype=ptype.arithmetic
)
Rank: 0
	b share:MPCTensor(
	_tensor=tensor([9153487009650949638, 7350214690636796293, 6393524024898254135,
           5694437736803597])
	plain_text=HIDDEN
	ptype=ptype.arithmetic
)
Rank: 1
	b share:MPCTensor(
	_tensor=tensor([-9153487009650925368, -7350214690636595554, -6393524024898213922,
           -5694437736725595])
	plain_text=HIDDEN
	ptype=ptype.arithmetic
)
Rank: 0
	aggregated:MPCTensor(
	_tensor=tensor([ -47048185290180,   25307633730810, -113907385499464,  137460525666116])
	plain_text=HIDDEN


[None, None]

In [37]:
@mpc.run_multiprocess(world_size=2)
def examine_arithmetic_shares():
    x_enc = crypten.cryptensor([1, 2, 3], ptype=crypten.mpc.arithmetic)

    rank = comm.get().get_rank()
    print(f"Rank {rank}:\n {x_enc}")

x = examine_arithmetic_shares()

Rank 1:
 MPCTensor(
	_tensor=tensor([-1714998675057671160, -6441751786708971437,  2289916992161866161])
	plain_text=HIDDEN
	ptype=ptype.arithmetic
)Rank 0:
 MPCTensor(
	_tensor=tensor([ 1714998675057736696,  6441751786709102509, -2289916992161669553])
	plain_text=HIDDEN
	ptype=ptype.arithmetic
)



In [45]:
import io

def local_value(rank):
    crypten.print(f"AAAAAAA: {rank}")
    buffer = io.BytesIO()
    torch.save(torch.randint(low=1, high=100, size=(5,), dtype=torch.int), buffer )
    return buffer

In [16]:
@mpc.run_multiprocess(world_size=2)
def run_collaborative_mean():
    
    #already noised coeffiecients
    rank = comm.get().get_rank()
    if rank == 0:
        # Alice
        a = torch.Tensor([2, 4, 5 ,8])
        # Not secure noise generation
        torch.manual_seed(rank)
        noise = torch.randn(a.shape) 
        crypten.print(f"Rank: {rank}\n\t Alice adding noise: {noise}", in_order=True)
        a +=  noise
        torch.save(a,"/tmp/my_experiments/p_0.pt")
        c=4
    else:
        # Bob
        b = torch.Tensor([3,7,9,10])
        # Not secure noise generation
        torch.manual_seed(rank)
        noise = torch.randn(b.shape)
        crypten.print(f"Rank: {rank}\n\t Bob adding noise: {noise}", in_order=True)
        b =+ noise
        torch.save(b,"/tmp/my_experiments/p_1.pt")  
        c=6

    a_enc = crypten.load_from_party("/tmp/my_experiments/p_0.pt", src=ALICE)
    b_enc = crypten.load_from_party("/tmp/my_experiments/p_1.pt", src=BOB)
   
    c =c*2
    crypten.print(f"Rank: {rank}\n\t local executed code : {c}", in_order=True)
    
    boh = torch.load(f"/tmp/my_experiments/p_{rank}.pt")
    crypten.print(f"Rank: {rank}\n\t boh : {boh}", in_order=True)
    
    crypten.print(f"Rank: {rank}\n\ta share: {a_enc}", in_order=True)
    crypten.print(f"Rank: {rank}\n\tb share:{b_enc}", in_order=True)


    agg = (a_enc + b_enc)*0.5


    crypten.print(f"Rank: {rank}\n\taggregated:{agg}", in_order=True)
    crypten.print(f"Rank: {rank}\n\taggregated:{agg.get_plain_text()}", in_order=True)

run_collaborative_mean()

Rank: 0
	 Alice adding noise: tensor([ 1.5410, -0.2934, -2.1788,  0.5684])
Rank: 1
	 Bob adding noise: tensor([0.6614, 0.2669, 0.0617, 0.6213])
Rank: 0
	 local executed code : 8
Rank: 1
	 local executed code : 12
Rank: 0
	 boh : tensor([3.5410, 3.7066, 2.8212, 8.5684])
Rank: 1
	 boh : tensor([0.6614, 0.2669, 0.0617, 0.6213])
Rank: 0
	a share: MPCTensor(
	_tensor=tensor([-4688026951519696257, -1975019885977018269, -7837516545229879928,
         8903740494244155483])
	plain_text=HIDDEN
	ptype=ptype.arithmetic
)
Rank: 1
	a share: MPCTensor(
	_tensor=tensor([ 4688026951519928319,  1975019885977261182,  7837516545230064818,
        -8903740494243593943])
	plain_text=HIDDEN
	ptype=ptype.arithmetic
)
Rank: 0
	b share:MPCTensor(
	_tensor=tensor([-7662890500681175509,  5114403462891938145,  6714405339130731550,
         6182570066623497053])
	plain_text=HIDDEN
	ptype=ptype.arithmetic
)
Rank: 1
	b share:MPCTensor(
	_tensor=tensor([ 7662890500681218851, -5114403462891920652, -6714405339130727508,

[None, None]

In [133]:
import time


def add_skellam_noise(v, poisson_lam=1.0):
    shape = (*v.shape, 2)  # Two draws of Poisson
    seed = int(time.time() * 10**6)
    print(seed)
    torch.manual_seed(seed)
    poissons = torch.poisson(torch.tensor([poisson_lam, poisson_lam]).repeat(*shape[:-1], 1))
    print((poissons[..., 0] - poissons[..., 1]).to(v.dtype))
    return v + (poissons[..., 0] - poissons[..., 1]).to(v.dtype)

# Example usage
v = torch.tensor([1.0, 2.0, 3.0])
result = add_skellam_noise(v)
print(result)

1687351524958049
tensor([-3.,  0.,  0.])
tensor([-2.,  2.,  3.])


In [134]:
@mpc.run_multiprocess(world_size=2)
def run_collaborative_mean():
    
    #already noised coeffiecients
    rank = comm.get().get_rank()
    if rank == 0:
        # Alice
        a = torch.Tensor([2, 4, 5 ,8])
        # Not secure noise generation
        a =  add_skellam_noise(a)
        torch.save(a,"/tmp/my_experiments/p_0.pt")
        c=4
    else:
        # Bob
        b = torch.Tensor([3,7,9,10])
        # Not secure noise generation
        b = add_skellam_noise(b)
        torch.save(b,"/tmp/my_experiments/p_1.pt")  
        c=6

    a_enc = crypten.load_from_party("/tmp/my_experiments/p_0.pt", src=ALICE)
    b_enc = crypten.load_from_party("/tmp/my_experiments/p_1.pt", src=BOB)
    
    crypten.print(f"Rank: {rank}\n\ta share: {a_enc}", in_order=True)
    crypten.print(f"Rank: {rank}\n\tb share:{b_enc}", in_order=True)


    agg = (a_enc + b_enc)*0.5


    crypten.print(f"Rank: {rank}\n\taggregated:{agg}", in_order=True)
    crypten.print(f"Rank: {rank}\n\taggregated:{agg.get_plain_text()}", in_order=True)

run_collaborative_mean()

16873515785552621687351578555245

tensor([-2., -1.,  1., -1.])tensor([ 0., -1.,  0.,  0.])

Rank: 0
	 local executed code : 8
Rank: 1
	 local executed code : 12
Rank: 0
	 boh : tensor([0., 3., 6., 7.])
Rank: 1
	 boh : tensor([ 3.,  6.,  9., 10.])
Rank: 0
	a share: MPCTensor(
	_tensor=tensor([ 4138450675936687728, -1960423809974154838,   637438794397428115,
         7795834382105558486])
	plain_text=HIDDEN
	ptype=ptype.arithmetic
)
Rank: 1
	a share: MPCTensor(
	_tensor=tensor([-4138450675936687728,  1960423809974351446,  -637438794397034899,
        -7795834382105099734])
	plain_text=HIDDEN
	ptype=ptype.arithmetic
)
Rank: 0
	b share:MPCTensor(
	_tensor=tensor([ 6080823694690380448, -1168128476060745276,  7969565709737082554,
        -5880454719276089910])
	plain_text=HIDDEN
	ptype=ptype.arithmetic
)
Rank: 1
	b share:MPCTensor(
	_tensor=tensor([-6080823694690183840,  1168128476061138492, -7969565709736492730,
         5880454719276745270])
	plain_text=HIDDEN
	ptype=ptype.arithmetic
)
Ran

[None, None]